# Item 99: Consider `memoryview` and `bytearray` for Zero-Copy

Interactions with `bytes`

## Notes

-   Python requires extra effort to parallelise CPU-bound computation
    (See [Item 79](../../Chapter_09/Item_079/item_079.qmd) and [Item
    94](../Item_094/item_094.qmd))
-   But, can support high-throughput parallel I/O (See [Item
    68](../../Chapter_09/Item_068/item_068.qmd) and [Item
    75](../../Chapter_09/Item_075/item_075.qmd))
-   However, understanding the tools available and how to use them
    *without* leading to slow code can require some skill
-   For example, consider a media-streaming server
    -   Users don’t need to download a video in advance
    -   Users can move forward or backward within a video
-   We might have functions to implement this by converting a time-code
    to a index and returning the associated chunk of data

In [1]:
import os # for demo only

def timecode_to_index(video_id, timecode):
    # Returns byte offser in the video data
    return 0  # placeholder


def request_chunk(video_id, byte_offset, size):
    # Returns size bytes of video_id's data from the offset
    # simulate by returning random data
    return os.urandom(size)


video_id = 1
timecode = "01:09:14:28"
byte_offset = timecode_to_index(video_id, timecode)
size = (8**2)
video_data = request_chunk(video_id, byte_offset, size)

print(f"{video_id=}, {timecode=}, {byte_offset=}, {video_data=}")

video_id=1, timecode='01:09:14:28', byte_offset=0, video_data=b'\xd43:\x81W>pj\xb2JY\xd9n\xb6\xe7Dn\x86\xfbt\x00.%\xb2(\x1a\xd4\xed\x9fa\x06\x1b67\xa5*\x92xA\xd5<0=\x95\xdd\x87Mga\x03^\xe44\xe7\x02y9\xf7)g\x1f\xf5$\xbf'

-   How do we now implement the server-side handler that receives
    `request_chunk`
    -   Must then return the associated video data chunk
-   First we assume that the program is driven by an `asyncio` process
    (See [Item 76](../../Chapter_09/Item_076/item_076.qmd))
    -   Now want to focus on how to handle extracting the chunk
    -   Assume video is cached memory
    -   Extracted then sent over a socket back to a client

In [2]:
import os # for demo only

def timecode_to_index(video_id, timecode):
    # Returns byte offser in the video data
    return 0  # placeholder


def request_chunk(video_id, byte_offset, size):
    # Returns size bytes of video_id's data from the offset
    return video_data[byte_offset : byte_offset + size]

# Adding in the handling

# simulate a socket connection
class NullSocket:
    def __init__(self):
        self.handle = open(os.devnull, "wb")

    def send(self, data):
        self.handle.write(data)

socket = NullSocket() # represents client socket connection
size = (8 ** 2) # Requested chunk size
video_data = os.urandom(20 * size) # Bytes containing data for video_id

video_id = 1
timecode = "01:09:14:28"

byte_offset = timecode_to_index(video_id, timecode)
chunk = request_chunk(video_id, byte_offset, size)
socket.send(chunk)

print(f"Sent {chunk=} over socket")

Sent chunk=b"\x91{\xf8~\x1f<\xff'\xce\xd7\xa5&\xf5KD\xdd\xd2.\xd4\x9a6\x00\x0b\x87\x1c\xec\x9a\xfb\x9a\x8a\xf7f\xe6\xfc\xdf\xb4\xd5\x8eef\x0e\x03\xcf\xf4l\x80xI\xdc\x87\xf9K|\xda\x95\x12\xd2\xc83i\xd6R\xc5)" over socket

-   Latency and throughput determined by two factors
    1.  How long to slice the chunk from `video_data`
    2.  How long to transmit over a socket
-   Focusing just on point 1, we can microbenchmark how long fetching a
    chunk takes.
    -   We’ll also exclude the function call wrapper
    -   Here we’ll set the size to $20$ MB.

In [3]:
import timeit

size = 20 * (1024**2)
video_data = os.urandom(20 * size) # Bytes containing data for
byte_offset = 0

def run_test():
    chunk = video_data[byte_offset : byte_offset + size]

result = ( timeit.timeit(stmt="run_test()", globals=globals(), number=100) / 100 )

print(f"{result:0.9f} seconds")

0.000936724 seconds

-   This takes about $5$ milliseconds
-   Theoretical server maximum throughput is thus, limited by video
    extraction speed as

$$
\begin{align}
    \frac{20 \text{ MB}}{5 \text{ ms}} &= 4 \text{ GB}\text{s}^{-1}
\end{align}
$$

-   Server also limited to,

$$
\begin{align}
    \frac{1 \text{ CPU=second}}{5 \text{ ms}} &= 200 \text{ clients in parallel}
\end{align}
$$

-   But we already know that `asyncio` should be able to scale up to
    tens of thousands of simultaneous connections
-   The slowdown is because as discussed slices create copies
    -   Copying consumes CPU time
-   Instead we can use `memoryview`
    -   A built-in type for handling the CPython `buffer` protocol
        -   Low-level C API allowing Python runtime and C extensions
            (See [Item 96](../Item_096/item_096.qmd)) to access
            underlying data buffers
            -   Can then treat them as `bytes` instances
        -   Since Python 3.12 the buffer protocol is also emulatable in
            python
-   `memoryview` can be sliced to create a new `memoryview` without a
    copy

In [4]:
data = b"shave and a haircut, two bits"
view = memoryview(data)
chunk = view[12:19]

print(chunk)
print("Size:            ", chunk.nbytes)
print("Data in view:    ", chunk.tobytes())
print("Underlying data: ", chunk.obj)

Size:             7
Data in view:     b'haircut'
Underlying data:  b'shave and a haircut, two bits'

-   These *zero-copy* operations can significantly speed-up code that
    heavily processes memory, e.g.
    1.  I/O-bound access
    2.  Heavy numerical mathematics (e.g. Numpy)
-   Using `memoryview` as a drop-in replacement for our video serving
    service

In [5]:
import timeit

size = 20 * (1024**2)
video_data = os.urandom(20 * size) # Bytes containing data for
video_view = memoryview(video_data)
byte_offset = 0

def run_test():
    chunk = video_view[byte_offset : byte_offset + size]

result = ( timeit.timeit(stmt="run_test()", globals=globals(), number=100) / 100 )

print(f"{result:0.9f} seconds")

0.000000400 seconds

-   This should run in a several hundred nanoseconds
-   So an order of magnitude faster than the `bytes` slicing technique
-   Our new theoretical maximum throughput is then

$$
\begin{align}
\frac{20 \text{ MB}}{250 \text{ ns}} &= 80 \text{ TB}\text{s}^{-1}
\end{align}
$$

-   Or in terms of parallel clients

$$
\begin{align}
\frac{1 \text{ CPU-second}}{250 \text{ ns}} &= 4 \times 10^{9}
\end{align}
$$

-   So four million clients. Now the program should be bound by the
    socket performance rather than CPU constraints.

-   Now consider a reversed process

    -   Users must submit live video streams that are then broadcast out
        to viewers

-   We need to store incoming video data

    -   Cache it for clients to read from

In [6]:
import os

def timecode_to_index(video_id, timecode):
    # Returns byte offser in the video data
    return 0  # placeholder

# socket connection from client


size = (4 ** 2) # Incoming chunk size
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]

video_id = 1
timecode = "01:09:14:28"
byte_offset = timecode_to_index(video_id, timecode) # Incoming buffer position
video_view = memoryview(video_cache)


class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

socket = MockIncomingSocket()
chunk = socket.recv(size)
before = video_view[:byte_offset]
after = video_view[byte_offset + size:]

new_cache = b"".join([before, chunk, after])

print(f"Updated the cache: {new_cache=}")

Updated the cache: new_cache=b"\x15\xa9\xfaI\x06\\v^w\x887\xbd\xcf\xe8\xean\xa4\x88d\xbe\x89-\xe5A\x8b^*K\xf5b[Z\xcf\x83\x8b\xd9<\xa7{TkB\xf8\xd1\x0e\xf8\x8e/\xf1E,'\xe6o*\xbaRi\x84\xb7J\x16NJ\x8fju\x99\xed\xbc\x9f\xc3\xdcd\xae\xe4C!\xba\x03|\x05\xbd\xcd?\xc0\x8d\xbf\xb7\xbeD\x16P\xaa\xf7&\x87\x00\xfcs\x1a\xe6j1\xbf\xdc+\xd5G&\xbfl\xbcj\x93\x91.j\x8b\xfeQ5O\x89}\x9fF\x1ct\xfb\xb7 +\x87}\x8e\xc8\xb3\xda*\x15{u\x80*\xa8\xe0w\xfc\xd3\tA\x8a\x0f\xef\xa0K\x8f/\xbf\x89\xbe\xac@=s\x88\x95\x1a9:\x99\x846\xe1\xcd\xc1\xb7|\xee\xf1\xaf\x87)\x81,\xb8Z0\xa2\x04\xcfO\x15\xb9hB'\x97\xf2\xc1\x92\xa0\xa0\xafU\xfa\x12t\x13,U\xd9\x85\xf0\xfa\x0cw\x8ek\xb0$^n\xdd\x03\xae\x05\xb5\x7f\xca\xfdA\x8d\xac\x87X\xa8\xbe\xc5l\x82lf\xb0>\xc0\xe0?Yr?N=\x9ep\x83\xb8\xcf\xd2i&\x19^\x9b\xbbf\x02\x17\xc7p2h\x8cW1\xeca.\x0e\xeb\xb1\xdb\xf7\x0b~\x98\x8at\xe7\x91B\xa8\x85\xd0\x84\xd8\x86\xe0l\xba\xd0\xcb\xb3\xd7\xfdo#ieJU\x13\t\xfe\xcbwzkk"

-   `socket.recv` returns a `bytes` instance
    -   Splice this into the existing cache
    -   Insert at the current `byte_offset` via slicing and `bytes.join`
-   Now need to profile the timing

In [7]:
import timeit
import os

class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

socket = MockIncomingSocket()
size = (1024 ** 2) # Incoming chunk size
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]
video_view = memoryview(video_cache)
byte_offset = 1234 # pick arbitrary point in the middle

def run_test():
    chunk = socket.recv(size)
    before = video_view[:byte_offset]
    after = video_view[byte_offset + size : ]
    new_cache = b"".join([before, chunk, after])

result = (timeit.timeit(stmt="run_test()", globals=globals(), number=100,) / 100)

print(f"{result:0.9f} seconds")

0.000930728 seconds

-   This takes about three milliseconds to receive $1$ MB and update the
    cache.
-   Maximum throughput to receive is then

$$
\begin{align}
\frac{1 \text{ MB}}{ 3 \text{ ms}} &\approx 330 \text{ MB}\text{s}^{-1}
\end{align}
$$

-   Means we are limited to about $300$ simultaneously streaming clients
-   Can use `bytearray` instead of `memoryview`
    -   `bytes` are immutable like strings

In [8]:
some_bytes = b"hello"
some_bytes[0] = 0x79

-   `bytearray` is effectively a mutable version of `bytes`
    -   Can overwrite indices
-   `bytearray` values are integers rather than bytes

In [9]:
array = bytearray(b"hello")
array[0] = 0x79
print(array)

bytearray(b'yello')

-   Can still wrap a `bytearray` in a `memoryview` to avoid extra copies
    -   Then can slice the `memoryview` and modify to overwrite the
        underlying `bytearray`

In [10]:
array = bytearray(b"row, row, row your boat")
view = memoryview(array)
write_view = view[3:13]
write_view[:] = b"-10 bytes-"
print(array)

bytearray(b'row-10 bytes- your boat')

-   Library methods in Python user the buffer protocol for fast data
    receipt or reading, e.g.
    1.  `socket.recv_into`
    2.  `RawIOBase.read_into`
-   These methods avoid creating copies and allocating memory
    -   Received data goes into existing buffer
-   We can convert our program to use `recv_into` and a `memoryview`
    slice to speed up our broadcasting method

In [11]:
class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

# socket connection from client
socket = MockIncomingSocket()

size = (4 ** 2) # Incoming chunk size
byte_offset = 1234
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]
video_view = memoryview(video_cache)

video_array = bytearray(video_cache)
write_view = memoryview(video_array)

chunk = write_view[byte_offset : byte_offset + size]
socket.recv_into(chunk)

print(f"New cache: {video_cache=}")

New cache: video_cache=b'\xf2"\xc8\'\xddO\xec\xeb\xee+Pls\xfeJ\x9cx\xae\x13\x07\xf9c\xbb\t\xf9\xe9\x01\xab\x08R\xa8\x08\xa2\xe4\xad\xa2\x07\x12\xef\x1cr\xd2\xaddZt\x13|\x02A\xe7n\xdc+$p\\\x87\x01\xf3\xbck\xbb\x04u\xb9\x8c\x8f7\xe8\xad\xa2\x1d#\x13S\xfa-l\xd7\x1e#T\xe6\x8d\\H\x7f\xd8\xeeg\xbe\xf0\x16N\x8f\xad\xe9\xb3~+u\xa4iA\x89n\xa3&i"\xec\xc5\x91!;\xd3\xe5[8\xa5\x968\x1e\xa9\xd9A\x06\xaf?#s\x1d\x040G\x9e\'\x0e-\x9f,\xdb\x05\xc7=t4\x1a\xfc\x16\x86\xa7\x19W\x03\xd6\x9e\xee$BN\xe3%!\xfe\x87|\xae:[\xbe\xc93kB8<\x8b\xcc@{\xe0\\\x9f\x91w\xa4\x11\xce\xa8%\xff\xa3\x8e\xa3\xd8\x01Z\x08@\xc0\xd1\x81\x8el\xbe)\xaf\x03\x96\xf6\x16\xc6}7\x9b\x03V\x19\xce\xf0%\x8b\x88\xad\xbfr?\xc7Akf\xffh\xde\x1e\x89\xa6C\xcev;=\x11\x96\xbag\xf0\xd2\xb4C~Y\x8a\xce7\x03\x19\xd3\nSG\xe8\xf6TU\xa5D\xf4|c\x1b\x1c-P?\x9b\xe9\xfc\x0c\xcfGF\xd3\xf4\xd1NE>\x7f\xa05\xa5v\xe8qF\xfd\xffc\xb5\xaa \xc6U\xa3{\xca\xee\xd6\x04\xd3\xb5\x97q\xc8D\xd1V'

-   We can again microbenchmark the result for a $1$ MB chunk

In [12]:
import timeit
import os

class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

# socket connection from client
socket = MockIncomingSocket()

size = (1024 ** 2) # Incoming chunk size
byte_offset = 1234
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]
video_view = memoryview(video_cache)

video_array = bytearray(video_cache)
write_view = memoryview(video_array)


def run_test():
    chunk = write_view[byte_offset : byte_offset + size]
    socket.recv_into(chunk)

result = (
    timeit.timeit(stmt="run_test()", globals=globals(), number=100) / 100
)

print(f"{result:0.9f} seconds")

0.000035933 seconds

-   On my machine this takes about $90 \;\mu\text{s}$. Which means we
    could support,

$$
\begin{align}
    \frac{1 \text{ MB}}{90 \; \mu\text{s}} &= 11 \text{ GB}\text{s}^{-1}
\end{align}
$$

-   Which also supports,

$$
\begin{align}
    \frac{11 \text{ GB}}{1 \text{MB}} &= 11,000 \text{ processes}
\end{align}
$$

-   Much better scalability

## Things to Remember

-   `memoryview` provides zero-copy methods for reading and writing to
    slices of objects supporting the buffer protocol
-   `bytearray` built-in provides a mutable `bytes`-like type
    -   Can be used for zero-copy data reads
    -   Works with functions like `socket.recv_into`
-   `memoryview` can wrap a `bytearray`
    -   Let’s received data to be spliced into an existing buffer
    -   No need for extra copies